# FYOE — train v4.3 model on Colab T4

**Before you start:** click `Runtime` → `Change runtime type` → set hardware accelerator to **T4 GPU**.

Then `Runtime` → `Run all`. ~20-25 min total. At the end you get `saved_model_v4.zip` to download.

**What changed (v4.0 → v4.3):**

v3 hit 99.06% IID but only **52.4% seed suite**. v4.0 fixed the biggest gaps (negation, past tense, multi-intent, contact recall). v4.2 targeted all 15 remaining seed failures with contrastive banks. **v4.3** adds:

- **Focal loss** — downweights easy examples, forces the model to focus on hard cases
- **Finer per-intent thresholds** — 0.01 grid instead of 0.05 (91 vs 17 candidates)
- **Question-form intents** — "can you venmo me?" (real people ask, they don't just command)
- **Idiomatic false positives** — "playing it safe" ≠ music, "good call" ≠ contact, "pay attention" ≠ money, "noted" ≠ note (90+ patterns)
- **Ultra-short boundary** — "uber please" fires, bare "uber" doesn't
- **Present-tense activity** — "I'm watching tv" ≠ video command
- **Third-person gossip** — "she paid 200 for that bag" ≠ money action
- **Label smoothing** — prevents overconfidence on easy examples
- **Mixed precision (fp16)** — 2x faster training

Total dataset: ~28K examples. Seed suite: 112 cases (was 82).

**Goal:** seed-suite ≥95% with IID ≥98%.

## 1. Setup — fresh VM, clone repo, install deps

**Before running:** push your v4 changes (`generate_data.py` import + integration block, `v4_failure_modes.py`) to a branch named `v4` on GitHub. The clone below pulls that branch.

In [ ]:
!nvidia-smi

In [ ]:
%cd /content
!rm -rf paychat-model
# Clone v4 branch — fall back to v3 if v4 doesn't exist yet so the notebook still runs.
!git clone -b v4 https://github.com/Akash-Cheerla/paychat-model.git || git clone -b v3 https://github.com/Akash-Cheerla/paychat-model.git
%cd paychat-model
!git log -1 --oneline
!ls training/v4_failure_modes.py && echo 'v4 banks present' || echo 'WARNING: v4_failure_modes.py missing — push v4 branch first'

In [ ]:
# Pin transformers to last stable 4.x. transformers 5.0 has regressions in checkpoint loading.
!pip install -q 'transformers==4.46.3' 'tokenizers>=0.20,<0.21' sentencepiece scikit-learn
import torch, transformers
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('transformers', transformers.__version__)

## 2. Generate training data (v4)

~19K examples: v3 positives + v4 failure-mode banks + per-intent hard negatives. The script also prints v4 category counts so you can verify the import worked before training.

In [ ]:
%cd /content/paychat-model/training
!python generate_data.py

In [ ]:
# Sanity check: confirm v4 categories actually landed in the dataset.
import json
from collections import Counter
ds = json.load(open('full_dataset.json'))
v4_cats = Counter(d['category'] for d in ds if d['category'].startswith('v4_'))
print(f'Total examples: {len(ds)}')
print(f'\nv4 categories present:')
for c, n in v4_cats.most_common():
    print(f'  {c:<22} {n}')
assert v4_cats, 'v4 categories missing — generate_data.py did not import V4_* banks'
n_contact = sum(1 for d in ds if d['labels']['contact'] == 1)
print(f'\ncontact positives: {n_contact}  (v3 had ~600; target ≥1000 to fix 0% recall)')

## 3. Fine-tune RoBERTa-base (v4.3)

New training improvements over v4.0:
- **Focal loss** (gamma=2.0) — the model stops wasting gradient on easy examples and focuses on the hard seed-suite failures
- **Label smoothing** (0.03) — prevents sigmoid heads from being overconfident (reduces false positives)
- **Mixed precision (fp16)** — 2x faster on T4, allows 8 epochs in the same time as 5
- **Finer threshold grid** — 0.01 steps instead of 0.05 (91 candidates vs 17)
- **Early stopping** — saves best model, stops if val stagnates for 3 epochs after epoch 5

In [ ]:
!python train.py \
  --model roberta-base \
  --epochs 8 \
  --batch-size 32 \
  --max-len 128 \
  --focal-loss \
  --focal-gamma 2.0 \
  --label-smoothing 0.03 \
  --fp16

## 4. Inspect results

In [ ]:
import json
from pathlib import Path
model_dir = Path('/content/paychat-model/saved_model')
print('Files in saved_model/:')
for f in sorted(model_dir.iterdir()):
    size_kb = f.stat().st_size / 1024
    print(f'  {f.name:35} {size_kb:>10.1f} KB')

print('\nLearned per-intent thresholds:')
with open(model_dir / 'thresholds.json') as f:
    thresholds = json.load(f)
for intent, thr in thresholds.items():
    print(f'  {intent:<14} {thr:.2f}')

print('\nPer-intent test metrics:')
with open(model_dir / 'training_report.json') as f:
    report = json.load(f)
print(f"  test exact-match: {report['test_exact_match']:.2%}")
print(f"  test hamming:     {report['test_hamming']:.2%}")
print()
print(f"  {'intent':<14} {'precision':>10} {'recall':>8} {'f1':>7}")
for intent, m in report['per_intent'].items():
    print(f"  {intent:<14} {m['precision']:>9.1%} {m['recall']:>7.1%} {m['f1']:>6.1%}")

## 5. Seed-suite regression (the real test)

v3 scored **43/82 (52.4%)**. v4.0 scored **67/82 (81.7%)**. v4.3 should reach **≥105/112 (≥94%)** on the expanded 112-case suite. This number is what actually matters.

If v4.3 doesn't beat 90% here, do not ship. Look at the failure list, expand the relevant bank, regenerate, retrain.

In [ ]:
%cd /content/paychat-model
!python eval/run_seed_baseline.py 2>&1 | tail -20

In [ ]:
# Pull the headline numbers from the report so they're easy to compare to v3.
import json
rpt = json.load(open('/content/paychat-model/eval/baseline_report.json'))
print(f"v4 seed-suite: {rpt['passed']}/{rpt['total']} passed ({rpt['passed']/rpt['total']*100:.1f}%)")
print(f"v4 IID test:   {rpt['test_exact_match']:.2f}% exact match")
print(f"v4 latency:    {rpt['ms_per_case']:.0f} ms/case")
print()
print('By tag:')
for tag, info in sorted(rpt['by_tag'].items(), key=lambda x: x[1]['passed']/x[1]['total']):
    rate = info['passed']/info['total']*100
    print(f"  {tag:<25} {info['passed']:>2}/{info['total']:<2}  ({rate:.0f}%)")

## 6. Sanity check — fire real messages through the trained model

In [ ]:
import torch, json
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = '/content/paychat-model/saved_model'
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
mdl = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval().cuda()
with open(f'{MODEL_DIR}/thresholds.json') as f:
    thresholds = json.load(f)
labels = list(mdl.config.id2label.values()) if mdl.config.id2label else list(thresholds.keys())

# Mix of v3-passing samples + v4 failures + NEW v4.3 edge cases.
samples = [
    # v3 broke on these:
    'call dad',                                     # contact
    'text mom',                                     # contact
    'call him',                                     # contact
    "I'm not paying for pizza",                     # silent (negation)
    'paid rent already',                            # silent (past)
    'set it for 6',                                 # silent (ambiguous)
    'send 20',                                      # silent (ambiguous)
    'uber to JFK at 5am tomorrow',                  # ride+maps (not alarm)
    'yaar venmo me 20 for pizza',                   # money (code-mixed)
    'mom ko phone karna hai',                       # contact (code-mixed)
    "don't forget to call mom thursday",             # contact+alarm
    'how much did I spend on food this month',      # silent (query)
    # v4.2 targets:
    'doordash it bro',                              # food_order (slang)
    'spotify it pls',                               # music (slang)
    'split the dinner bill 4 ways',                 # money (split)
    'set a reminder',                               # alarm (generic)
    # v4.3 NEW — question forms:
    'can you venmo me 30',                          # money (question)
    'should we order pizza',                        # food_order (question)
    'wanna uber there',                             # ride (question)
    # v4.3 NEW — idiomatic false positives:
    'playing it safe',                              # silent
    'good call',                                    # silent
    'pay attention',                                # silent
    'noted',                                        # silent
    'watch your back',                              # silent
    'riding the wave',                              # silent
    "that's a tall order",                          # silent
    # v4.3 NEW — present tense / third person:
    "I'm watching tv",                              # silent
    "she paid 200 for that bag",                    # silent (third person)
    "he ordered food from doordash",                # silent (third person)
    # v4.3 NEW — ultra short boundary:
    'uber please',                                  # ride
    'venmo me',                                     # money
    'uber',                                         # silent (bare word)
    'venmo',                                        # silent (bare word)
    # regressions — must still pass:
    'venmo me 20 bucks for pizza',
    'remind me to call mom tomorrow at 6pm',
    'flight to Tokyo next month',
    'I love Paris',                                 # silent
    'watched Stranger Things last week',            # silent
]

for text in samples:
    enc = tok(text, return_tensors='pt', truncation=True, max_length=128).to('cuda')
    with torch.no_grad():
        logits = mdl(**enc).logits[0]
    probs = torch.sigmoid(logits).cpu().tolist()
    fired = [(labels[i], probs[i]) for i in range(len(labels)) if probs[i] >= thresholds.get(labels[i], 0.5)]
    fired.sort(key=lambda x: -x[1])
    fired_str = ', '.join(f'{l}={p:.2f}' for l, p in fired) or '(none)'
    print(f'{text!r:<55} -> {fired_str}')

## 7. Zip and download `saved_model/`

Drop the unzipped folder into your local repo at `./saved_model/`, then re-run `python eval/run_seed_baseline.py` locally to refresh `eval/baseline_report.md`. Compare to the v3 baseline you already have on disk.

In [ ]:
%cd /content/paychat-model
!zip -r saved_model_v4.zip saved_model/ -x '*.bin.tmp' '*.cache*' > /dev/null
!ls -lh saved_model_v4.zip
from google.colab import files
files.download('saved_model_v4.zip')